# 💾 Model Selection & Persistence
## Save Best Model as Pickle

This notebook:
- **Loads** previous model comparison results
- **Selects** the best performing model
- **Retrains** it on the full dataset
- **Creates** model metadata and documentation
- **Saves** the model in pickle format
- **Tests** model loading and predictions

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
from datetime import datetime
import os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

## Section 1: Load Model Comparison Results

In [ ]:
print("=" * 90)
print("MODEL SELECTION & PERSISTENCE")
print("=" * 90)

# Load model comparison results
results_file = r'C:\Aarthi\Guvi\MINI_PROJECT_3\model_comparison_results.csv'

try:
    results_df = pd.read_csv(results_file, index_col=0)
    print(f"\n✓ Loaded model comparison results")
    print(f"\nModels Evaluated:")
    for idx, (model_name, row) in enumerate(results_df.iterrows(), 1):
        print(f"  {idx}. {model_name:<35} Test R²={row['Test_R²']:.4f} RMSE=${row['Test_RMSE']:.2f}")
except Exception as e:
    print(f"⚠ Error loading results: {e}")
    results_df = None

## Section 2: Select Best Model

In [ ]:
print("\n" + "=" * 90)
print("MODEL SELECTION")
print("=" * 90)

# Select best model by R² score (primary metric)
best_model_name = results_df['Test_R²'].idxmax()
best_r2 = results_df.loc[best_model_name, 'Test_R²']
best_rmse = results_df.loc[best_model_name, 'Test_RMSE']
best_mae = results_df.loc[best_model_name, 'Test_MAE']
best_mse = results_df.loc[best_model_name, 'Test_RMSE'] ** 2

print(f"\n🏆 BEST MODEL SELECTED: {best_model_name}")
print(f"\nPerformance Metrics:")
print(f"  R² Score: {best_r2:.4f} (explains {best_r2*100:.2f}% of variance)")
print(f"  RMSE: ${best_rmse:.2f} (average prediction error)")
print(f"  MAE: ${best_mae:.2f} (typical error)")
print(f"  MSE: {best_mse:.4f}")

## Section 3: Retrain Best Model on Full Dataset

In [ ]:
print("\n" + "=" * 90)
print("RETRAINING BEST MODEL ON FULL DATASET")
print("=" * 90)

# Load data
df = pd.read_csv(r'C:\Aarthi\Guvi\MINI_PROJECT_3\taxi_fare_transformed.csv')

# Separate features and target
target = 'fare_amount' if 'fare_amount' in df.columns else df.columns[-1]
X = df.drop(columns=[target]) if target in df.columns else df.iloc[:, :-1]
y = df[target] if target in df.columns else df.iloc[:, -1]

print(f"\nLoading data: {X.shape[0]:,} samples, {X.shape[1]} features")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features scaled successfully")

# Train best model on full dataset
print(f"\nTraining {best_model_name} on full dataset...")
if best_model_name == 'Linear Regression':
    best_model = LinearRegression()
elif best_model_name == 'Ridge Regression':
    best_model = Ridge(alpha=1.0, random_state=42)
elif best_model_name == 'Lasso Regression':
    best_model = Lasso(alpha=0.01, random_state=42)
elif best_model_name == 'ElasticNet Regression':
    best_model = ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42)
elif best_model_name == 'Gradient Boosting':
    best_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
else:
    best_model = LinearRegression()

best_model.fit(X_scaled, y)
print(f"✓ {best_model_name} trained successfully")

## Section 4: Create Model Metadata

In [ ]:
print("\n" + "=" * 90)
print("CREATING MODEL METADATA")
print("=" * 90)

model_metadata = {
    'model_name': best_model_name,
    'model_type': 'Regression',
    'task': 'Taxi Fare Prediction',
    'training_date': datetime.now().isoformat(),
    'version': '1.0',
    'performance_metrics': {
        'R2_score': float(best_r2),
        'MSE': float(best_mse),
        'RMSE': float(best_rmse),
        'MAE': float(best_mae),
    },
    'feature_names': X.columns.tolist(),
    'feature_scaling': {
        'method': 'StandardScaler (z-score normalization)',
        'mean': scaler.mean_.tolist(),
        'std': scaler.scale_.tolist(),
    },
    'training_data': {
        'samples': int(X.shape[0]),
        'features': int(X.shape[1]),
        'target_variable': target,
    },
    'target_statistics': {
        'mean': float(y.mean()),
        'std': float(y.std()),
        'min': float(y.min()),
        'max': float(y.max()),
    }
}

print(f"✓ Metadata created successfully")

## Section 5: Save Model as Pickle

In [ ]:
print("\n" + "=" * 90)
print("SAVING MODEL & METADATA")
print("=" * 90)

# Save model to pickle
model_pickle_path = r'C:\Aarthi\Guvi\MINI_PROJECT_3\best_taxi_fare_model.pkl'
try:
    with open(model_pickle_path, 'wb') as f:
        pickle.dump(best_model, f)
    file_size = os.path.getsize(model_pickle_path)
    print(f"\n✓ Model saved to pickle")
    print(f"  File: best_taxi_fare_model.pkl")
    print(f"  Size: {file_size:,} bytes")
except Exception as e:
    print(f"✗ Error saving model: {e}")

# Save scaler
scaler_path = r'C:\Aarthi\Guvi\MINI_PROJECT_3\model_scaling_params.pkl'
try:
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"\n✓ Scaler saved to pickle")
    print(f"  File: model_scaling_params.pkl")
except Exception as e:
    print(f"✗ Error saving scaler: {e}")

# Save metadata
metadata_path = r'C:\Aarthi\Guvi\MINI_PROJECT_3\model_metadata.json'
try:
    with open(metadata_path, 'w') as f:
        json.dump(model_metadata, f, indent=2)
    print(f"\n✓ Metadata saved to JSON")
    print(f"  File: model_metadata.json")
except Exception as e:
    print(f"✗ Error saving metadata: {e}")

## Section 6: Test Model Loading & Predictions

In [ ]:
print("\n" + "=" * 90)
print("TESTING MODEL LOADING & PREDICTIONS")
print("=" * 90)

try:
    # Load model
    print("\nLoading model from pickle...")
    with open(model_pickle_path, 'rb') as f:
        loaded_model = pickle.load(f)
    print("✓ Model loaded successfully")
    
    # Load scaler
    print("Loading scaler...")
    with open(scaler_path, 'rb') as f:
        loaded_scaler = pickle.load(f)
    print("✓ Scaler loaded successfully")
    
    # Test predictions
    print("\nTesting predictions on sample data...")
    sample_indices = [0, 100, 1000, 10000, -1]
    
    for idx in sample_indices:
        if abs(idx) < len(X):
            X_sample = X.iloc[idx:idx+1].values
            X_sample_scaled = loaded_scaler.transform(X_sample)
            
            y_true = y.iloc[idx]
            y_pred = loaded_model.predict(X_sample_scaled)[0]
            error = abs(y_true - y_pred)
            
            print(f"\n  Sample {idx}:")
            print(f"    Actual: ${y_true:.2f}")
            print(f"    Predicted: ${y_pred:.2f}")
            print(f"    Error: ${error:.2f} ({100*error/y_true:.1f}%)")
    
    print("\n✓ Model loading and prediction test successful!")
    
except Exception as e:
    print(f"✗ Error during testing: {e}")

print("\n" + "=" * 90)
print("✅ MODEL SELECTION & PERSISTENCE COMPLETE!")
print("=" * 90)